In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, bronze

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
key = common.match_key(env)
tables = bronze.load_tables()
print(cfg["code"], env, len(tables), "tables")

In [0]:
landing = bronze.list_landing(spark, cfg)
total = loaded = rejected = 0
for table, spec in tables.items():
    rows, ok, bad = bronze.load_table(spark, cfg, table, spec, key, landing)
    total, loaded, rejected = total + rows, loaded + ok, rejected + bad
    print(f"{table:20} +{rows:<6} files {ok:<4} rejected {bad}")
common.log_run(spark, cfg, "bronze", "success", total, f"{loaded} files loaded, {rejected} rejected")
print("TOTAL rows", total, "| files loaded", loaded, "| rejected", rejected)